In [1]:
import pandas as pd
from pymongo import MongoClient
from IPython.display import display  # 用於在 Notebook 中更美觀地顯示 DataFrame

In [2]:
# MongoDB 連線資訊 (請替換成你的實際資訊)
mongo_uri = 'mongodb://localhost:27017/'  # 預設本機連線
database_name = 'mydatabase'         # 你的資料庫名稱
collection_name = 'reviews'        # 你的 Collection 名稱

try:
    client = MongoClient(mongo_uri)
    db = client[database_name]
    collection = db[collection_name]
    print(f"成功連線到 MongoDB 資料庫 '{database_name}'，Collection: '{collection_name}'")
except Exception as e:
    print(f"連線 MongoDB 失敗: {e}")
    exit()

成功連線到 MongoDB 資料庫 'mydatabase'，Collection: 'reviews'


## 提取資料庫中所有的 ProductId

已確認非 B 開頭的 ProductId 的商品僅有 8 個，為求方便直接忽略。

In [3]:
# 從 collection 中提取所有獨立的 ProductId
unique_product_ids = collection.distinct("ProductId")

# 將其轉換為 Pandas DataFrame
df_unique_product_ids = pd.DataFrame(unique_product_ids, columns=["ProductId"])

# 篩選出以 "B" 開頭的商品 ID
df_b_products = df_unique_product_ids[df_unique_product_ids['ProductId'].str.startswith('B')]

# 顯示篩選結果
display(df_b_products)

,ProductId
8,B00002N8SM
9,B00002NCJC
10,B00002Z754
11,B00004CI84
12,B00004CXX9
...,...
74253,B009UOFTUI
74254,B009UOFU20
74255,B009UUS05I
74256,B009WSNWC4


## Get Product Metadata for other datasets

https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

In [4]:
import urllib.request
import os

url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/all_categories.txt"
filename = "all_categories.txt"
save_directory = "./"

filepath = os.path.join(save_directory, filename)

if not os.path.exists(filepath):
    try:
        urllib.request.urlretrieve(url, filepath)
        print(f"檔案 '{filename}' 下載完成，儲存於 '{save_directory}'。")
    except Exception as e:
        print(f"下載失敗: {e}")
else:
    print(f"檔案 '{filename}' 已存在於 '{save_directory}'。")

with open(filename, 'r', encoding='utf-8') as f:
    categories = f.readlines() # 將每一行讀取到一個 list 中
    # print(f"檔案 '{filename}' 內容:")
    # for line in categories:
    #     print(line.strip())  # 去掉行尾的換行符號

檔案 'all_categories.txt' 下載完成，儲存於 './'。


In [5]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm # Optional: for progress bar
import gc # Garbage Collector
import json # To handle potential JSON string parsing if needed later


# 將我們要查找的本地 ProductId 存儲在一個集合中，以便快速查找
local_product_ids_set = set(df_b_products['ProductId'])
print(f"已載入 {len(local_product_ids_set)} 個以 'B' 開頭的本地 ProductId 供比對。")

# --- 初始化一個字典來收集所有匹配到的商品記錄 (以 parent_asin 為鍵，確保唯一性) ---
all_matched_records_dict = {}

# --- 迭代處理每個類別 ---
for category in categories:
    category = category.strip()
    print(f"\n--- 開始處理類別: {category} ---")
    
    dataset_stream = None # 初始化變數
    matched_count_in_category = 0
    
    try:
        # 1. 使用串流模式載入資料集
        print(f"以串流模式載入資料集: raw_meta_{category}...")
        dataset_stream = load_dataset(
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_meta_{category}", # Use f-string for clarity
            split="full", # 確保 'full' split 存在
            streaming=True,      # <--- 啟用串流模式
            trust_remote_code=True
        )

        # 2. 迭代串流資料集，提取 parent_asin 並查找匹配項
        print("迭代串流資料集以查找匹配的商品記錄...")
        
        # 使用 tqdm 顯示進度條 (可選)
        for record in tqdm(dataset_stream, desc=f"處理 {category}"):
            # 確保 record 中有 'parent_asin' 鍵且值不是 None
            if record and 'parent_asin' in record and record['parent_asin'] is not None:
                dataset_asin = str(record['parent_asin']) # 確保是字串

                # 如果資料集的 ASIN 存在於我們的本地 ID 集合中
                if dataset_asin in local_product_ids_set:
                    # 檢查是否已記錄過此 ASIN，如果沒有，則添加記錄
                    # 這可以防止同一商品在不同類別的資料流中出現時被重複添加
                    if dataset_asin not in all_matched_records_dict:
                        # --- 直接儲存整個 record 字典 ---
                        all_matched_records_dict[dataset_asin] = record 
                        matched_count_in_category += 1
                    # 如果你希望即使在不同類別中看到也要更新記錄（可能性不大，但取決於你的邏輯），
                    # 可以取消上面的 if 檢查，直接賦值：
                    # all_matched_records_dict[dataset_asin] = record
                    # 但這樣可能會覆蓋來自先前類別的記錄


        print(f"類別 '{category}' 處理完成。在此類別中新找到 {matched_count_in_category} 個匹配的商品記錄。")
        print(f"目前累積找到 {len(all_matched_records_dict)} 個唯一的匹配商品記錄。")

    except Exception as e:
        print(f"處理類別 '{category}' 時發生錯誤: {e}")
        # continue # or break
        
    finally:
        # --- 清理資源 ---
        del dataset_stream 
        gc.collect() # 嘗試觸發垃圾回收
        print(f"已清理類別 '{category}' 的資源。")


# --- 所有類別處理完成後 ---
print("\n--- 所有類別處理完成 ---")

# 3. 將收集到的所有記錄轉換為 DataFrame
if all_matched_records_dict:
    print(f"總共找到 {len(all_matched_records_dict)} 個唯一的匹配商品記錄。")
    print("正在將收集到的記錄轉換為 DataFrame...")
    
    # 從字典的值（即記錄本身）創建列表，然後轉換為 DataFrame
    final_matched_records_list = list(all_matched_records_dict.values())
    final_matched_df = pd.DataFrame(final_matched_records_list)

    # --- 可選：處理嵌套結構 ---
    # Pandas 會將字典或列表的列創建為 object 類型。對於 CSV，它們會被轉換為字符串。
    # 如果你想在存儲前稍微處理一下（例如，將 images 字典展平成特定 URL），可以在這裡做。
    # 範例：提取主要圖片 URL
    # def get_main_image(img_dict):
    #     try:
    #         if isinstance(img_dict, dict) and 'large' in img_dict and img_dict['large']:
    #             return img_dict['large'][0]
    #     except Exception:
    #         pass
    #     return None
    # final_matched_df['main_image_url'] = final_matched_df['images'].apply(get_main_image)
    # final_matched_df = final_matched_df.drop(columns=['images']) # 可以選擇刪除原始嵌套列
    
    # 對於 'details' JSON 字符串，如果需要，可以解析它
    # def parse_details(details_str):
    #     try:
    #         return json.loads(details_str) # 返回字典
    #     except Exception:
    #         return details_str # 或返回 None 或空字典
    # final_matched_df['details_parsed'] = final_matched_df['details'].apply(parse_details)

    # 4. 將整合後的匹配結果（來自資料集的元數據）儲存為單一 CSV 文件
    output_filename = "matched_products_metadata.csv"
    print(f"將所有匹配商品的元數據 ({len(final_matched_df)} 筆) 儲存到: {output_filename}")
    
    try:
        # index=False 避免寫入 DataFrame 索引
        # encoding='utf-8' 確保兼容性
        final_matched_df.to_csv(output_filename, index=False, encoding='utf-8')
        print("CSV 文件儲存成功。")
    except Exception as e:
        print(f"儲存 CSV 文件時發生錯誤: {e}")

else:
    print("在所有處理的類別中，沒有找到任何匹配的商品記錄。")

print("\n--- 腳本執行完畢 ---")

/home/simslab/anaconda3/envs/mongo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


已載入 74250 個以 'B' 開頭的本地 ProductId 供比對。

--- 開始處理類別: All_Beauty ---
以串流模式載入資料集: raw_meta_All_Beauty...
迭代串流資料集以查找匹配的商品記錄...


處理 All_Beauty: 112590it [00:46, 2443.09it/s]


類別 'All_Beauty' 處理完成。在此類別中新找到 3 個匹配的商品記錄。
目前累積找到 3 個唯一的匹配商品記錄。
已清理類別 'All_Beauty' 的資源。

--- 開始處理類別: Toys_and_Games ---
以串流模式載入資料集: raw_meta_Toys_and_Games...
迭代串流資料集以查找匹配的商品記錄...


處理 Toys_and_Games: 890874it [08:13, 1806.23it/s]


...

類別 'Health_and_Personal_Care' 處理完成。在此類別中新找到 22 個匹配的商品記錄。
目前累積找到 33803 個唯一的匹配商品記錄。
已清理類別 'Health_and_Personal_Care' 的資源。

--- 開始處理類別: Appliances ---
以串流模式載入資料集: raw_meta_Appliances...
迭代串流資料集以查找匹配的商品記錄...


處理 Appliances: 94327it [01:08, 1380.10it/s]


類別 'Appliances' 處理完成。在此類別中新找到 10 個匹配的商品記錄。
目前累積找到 33813 個唯一的匹配商品記錄。
已清理類別 'Appliances' 的資源。

--- 開始處理類別: Movies_and_TV ---
以串流模式載入資料集: raw_meta_Movies_and_TV...
迭代串流資料集以查找匹配的商品記錄...


處理 Movies_and_TV: 748224it [05:24, 2305.40it/s]


類別 'Movies_and_TV' 處理完成。在此類別中新找到 4 個匹配的商品記錄。
目前累積找到 33817 個唯一的匹配商品記錄。
已清理類別 'Movies_and_TV' 的資源。

--- 所有類別處理完成 ---
總共找到 33817 個唯一的匹配商品記錄。
正在將收集到的記錄轉換為 DataFrame...
將所有匹配商品的元數據 (33817 筆) 儲存到: matched_products_metadata.csv
CSV 文件儲存成功。

--- 腳本執行完畢 ---
